## Data Collections

In [ ]:
import kaggle
import pandas as pd
import os
import csv

In [ ]:
# Télécharger le dataset depuis Kaggle
# !kaggle datasets download -d rishabhkausish/reddit-depression-dataset -p ../data/ --unzip

In [ ]:
# Chercher le fichier CSV extrait
csv_files = [f for f in os.listdir("../data/") if f.endswith('.csv')]
if csv_files:
    df = pd.read_csv(os.path.join("../data/", csv_files[0]))
    print("Dataset chargé dans le DataFrame 'df' est:", csv_files[0])
else:
    print("Aucun fichier CSV trouvé dans le dossier ../data/.")

In [ ]:
# Supprimer les lignes vides de la colonne Unnamed: 0
print("Nombre de valeurs manquantes dans la colonne 'Unnamed: 0' avant suppression :", df["Unnamed: 0"].isna().sum())
df = df.dropna(subset=["Unnamed: 0"])
print("Nombre de valeurs manquantes dans la colonne 'Unnamed: 0' après suppression :", df["Unnamed: 0"].isna().sum())

In [ ]:
# Supprimer les lignes vides de la colonne title
print("Nombre de valeurs manquantes dans la colonne 'title' avant suppression :", df["title"].isna().sum())
df = df.dropna(subset=["title"])
print("Nombre de valeurs manquantes dans la colonne 'title' après suppression :", df["title"].isna().sum())

In [ ]:
# 1) Copie de sécurité
df_backup = df.copy(deep=True)

# 2) Lignes où title contient au moins un chiffre
# mask = df["title"].astype(str).str.contains(r"\d", na=False) # Vérifie si la colonne "title" contient au moins un chiffre
unnamed_is_numeric = pd.to_numeric(df["Unnamed: 0"], errors="coerce").notna() # Vérifie si la colonne "Unnamed: 0" est numérique
title_is_numeric = pd.to_numeric(df["title"], errors="coerce").notna() # Vérifie si la colonne "title" est numérique
mask = (~unnamed_is_numeric) & title_is_numeric # On veut les lignes où "Unnamed: 0" n'est pas numérique et "title" est numérique

# 3) Inversion des valeurs entre les 2 colonnes sur ces lignes
df.loc[mask, ["Unnamed: 0", "title"]] = (
    df.loc[mask, ["title", "Unnamed: 0"]].to_numpy()
)

In [ ]:
# 1) Vérifier que Unnamed: 0 est bien numérique partout
print(f"Nombre de valeurs non numériques dans la colonne 'Unnamed: 0' : {pd.to_numeric(df['Unnamed: 0'], errors='coerce').isna().sum()}")

# 2) Vérifier que title est bien majoritairement non numérique
print(f"Nombre de titres numériques dans la colonne 'title' : {pd.to_numeric(df['title'], errors='coerce').notna().sum()}")

# 3) Si tout est ok, convertir définitivement Unnamed: 0 en int
# df["Unnamed: 0"] = df["Unnamed: 0"].astype(int)
df["Unnamed: 0"] = pd.to_numeric(df["Unnamed: 0"], errors="coerce").astype("Int64")

# 4) Rename the column "Unnamed: 0" to "id"
df = df.rename(columns={"Unnamed: 0": "id"})

# 5) Set the "id" column as the index of the DataFrame
df = df.set_index("id")

In [ ]:
# Supprime les lignes vides de la colonne "label"
print(f"Nombre de valeurs manquantes dans 'label' avant suppression : {df['label'].isna().sum()}") # Affiche le nombre de valeurs manquantes dans la colonne "label" avant suppression
df = df.dropna(subset=["label"])
print(f"Nombre de valeurs manquantes dans 'label' après suppression : {df['label'].isna().sum()}") # Affiche le nombre de valeurs manquantes dans la colonne "label" après suppression

In [ ]:
# Enregistrer le DataFrame nettoyé dans un nouveau fichier CSV
df.to_csv("../data/reddit_depression_dataset_cleaned_v1.csv", index=False)

In [ ]:
# Ouvre le CSV ../data/reddit_depression_dataset_cleaned_v1.csv
df = pd.read_csv("../data/reddit_depression_dataset_cleaned_v1.csv")

Points forts :

Recall 91% sur dépressifs : détecte 9/10 posts dépressifs (crucial en santé mentale)

Precision 76% : quand il dit "dépressif", c’est juste 3 fois sur 4

F1 83% : excellent compromis

# Données propres.
Pipeline de préparation pour NLP :

Étape 1 : EDA rapide

In [ ]:
# Distribution des labels
print(f"Distribution des labels :")
print(df["label"].value_counts(normalize=True) * 100)

In [ ]:
# Longueur des textes
df.loc[:, "text_length"] = df["title"].str.len() + df["body"].str.len()
print("Longueur textes :\n", df["text_length"].describe())

In [ ]:
# NaN restants
print(f"Nombre de valeurs manquantes dans le DataFrame : {df.isna().sum().sum()}")

In [ ]:
# Concat title + body (standard pour Reddit)
# df = df.assign(text=lambda x: x["title"].fillna("") + " " + x["body"].fillna(""))
df.loc[:, "clean_text"] = df["title"].fillna("") + " " + df["body"].fillna("")

# Nettoyage basique
import re
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'http\S+', '', str(text)) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', str(text)) # Remove punctuation and numbers
    return text.lower().strip() # Convert to lowercase and remove leading/trailing whitespace

df.loc[:, "clean_text"] = df["clean_text"].apply(preprocess_text)

In [ ]:
# Drop "title" and "body" columns
df = df.drop(columns=["title", "body"])

In [ ]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv("../data/reddit_depression_dataset_cleaned_v2.csv", index=False, quoting=csv.QUOTE_ALL)
print("✅ Dataset nettoyé sauvegardé !\nConcat title + body et nettoyage basique effectué dans la colonne 'clean_text'.")

In [ ]:
df.head()

In [ ]:
# 4. Sauvegarde
# df[["label", "clean_text"]].to_parquet("reddit_depression_clean.parquet")
# print("✅ Dataset nettoyé sauvegardé !")

In [ ]:
print(f"Distribution des labels :")
print(df["label"].value_counts())

# Dataset prêt avec ~2M posts non-dépressifs (80,8%) et ~480k dépressifs (19,2%).

1. Diagnostic du déséquilibre
Problème majeur : classe très déséquilibrée (80/20). Un modèle naïf qui prédit toujours "0" aurait déjà 80% d’accuracy.
​
On doit absolument gérer ça pour avoir un modèle utile

2. Premier modèle de base (BC02)
Voici le code complet d’entraînement d’un modèle baseline simple qui gère le déséquilibre :

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# 1. Split stratifié (même répartition train/test)
X = df["clean_text"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 2. Vectorisation TF-IDF (5000 features, bi-grammes)
vectorizer = TfidfVectorizer(
    max_features=5000, 
    stop_words='english', 
    ngram_range=(1,2),
    min_df=5
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
# 3. Modèle avec gestion déséquilibre (class_weight)
model = LogisticRegression(
    random_state=42, 
    max_iter=1000,
    class_weight='balanced'  # ⚠️ CLÉ pour le déséquilibre
)
model.fit(X_train_tfidf, y_train)

In [ ]:
# 4. Évaluation
y_pred = model.predict(X_test_tfidf)
print("Classification Report :\n", classification_report(y_test, y_pred))
print("Confusion Matrix :\n", confusion_matrix(y_test, y_pred))

In [ ]:
# 5. Sauvegarde (pour RNCP BC02)
joblib.dump(model, '../models/depression_classifier.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print("✅ Modèle baseline sauvegardé !")

In [ ]:
# Charger et tester
loaded_model = joblib.load('../models/depression_classifier.pkl')
loaded_vectorizer = joblib.load('../models/tfidf_vectorizer.pkl')

In [ ]:
test_text = "I feel very unhappy and hopeless about my life"  # Exemple de texte à tester
test_vec = loaded_vectorizer.transform([test_text])
pred = loaded_model.predict(test_vec)[0]
proba = loaded_model.predict_proba(test_vec)[0]

print(f"Texte : {test_text}")
print(f"Prédiction : {'Dépressif' if pred==1 else 'Non dépressif'}")
print(f"Probabilité dépressif : {proba[1]:.2%}")

Points forts :

Recall 91% sur dépressifs : détecte 9/10 posts dépressifs (crucial en santé mentale)

Precision 76% : quand il dit "dépressif", c’est juste 3 fois sur 4

F1 83% : excellent compromis